# 01 — Crosswalk universal das 6 UFs

Constrói o crosswalk canônico (2.363 munis × 5 chaves) a partir do
universo core validado. Output: `data/interim/crosswalk_centrosul.csv`.

Toda camada subsequente vai usar esse arquivo para chavear seus dados.

**Decisões (pré-registro v2.2 §3.2):**
- 6 UFs: SP, GO, MG, PR, MS, MT (sem ES)
- Total: 2.363 municípios = soma exata IBGE
- Chaves expostas: `geocode` (primária), `muni_key`, `cidade_uf_seeg`

In [ ]:
# Setup: monta Drive, adiciona pipeline ao sys.path
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/Renovabio - EcoEco')
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Forçar reload do módulo crosswalk
import importlib
from pipeline import crosswalk
importlib.reload(crosswalk)

from pipeline.crosswalk import (
    load_universo_core,
    build_crosswalk,
    validate_crosswalk,
    save_crosswalk,
    IBGE_REF_COUNTS,
)
print('✓ pipeline.crosswalk carregado')

## Inspeção do universo core

In [ ]:
# Lê o universo core (já validado em 00_setup) e mostra a estrutura
uc = load_universo_core()
print(f'Universo core: {uc.shape}')
print(f'Colunas: {list(uc.columns)}')
print()
print('Primeiras 5 linhas:')
uc.head()

## Constrói chaves canônicas

In [ ]:
# Constrói as 5 chaves canônicas: geocode, municipio, uf, cidade_uf_seeg, muni_key
cw = build_crosswalk()
print(f'Crosswalk: {cw.shape}')
print()
print('Colunas:')
for c in cw.columns:
    print(f'  - {c}')
print()
print('Amostra (uma de cada UF):')
amostra = cw.groupby('uf').head(1).sort_values('uf')
amostra

## Validação em 4 dimensões

In [ ]:
# Valida o crosswalk em 4 dimensões:
# 1. Total de linhas bate com IBGE (2.363)
# 2. Contagem por UF bate com IBGE oficial
# 3. geocode é único
# 4. muni_key é único
report = validate_crosswalk(cw)

print(f'OK: {report["ok"]}')
print(f'Total: {report["n_total"]}')
print(f'\nPor UF:')
for uf, n in sorted(report['by_uf'].items()):
    ref = IBGE_REF_COUNTS[uf]
    diff = n - ref
    flag = '✓' if diff == 0 else '⚠️'
    print(f'  {flag} {uf}: {n} (IBGE={ref}, diff={diff:+d})')

if report['errors']:
    print(f'\n⚠️  Erros encontrados:')
    for e in report['errors']:
        print(f'  - {e}')
else:
    print(f'\n✅ Validação passou em todas as 4 dimensões')

## Inspeção das chaves canônicas

In [ ]:
# Verifica se as 3 chaves derivadas estão bem formadas
print('=== Exemplo de chaves para São Paulo - SP ===\n')

ex = cw[cw['municipio'].str.contains('São Paulo', na=False, regex=False)]
ex = ex[ex['uf'] == 'SP'].iloc[0]

print(f'  geocode         : {ex["geocode"]!r}')
print(f'  municipio       : {ex["municipio"]!r}')
print(f'  uf              : {ex["uf"]!r}')
print(f'  cidade_uf_seeg  : {ex["cidade_uf_seeg"]!r}')
print(f'  muni_key        : {ex["muni_key"]!r}')

print(f'\n=== Cidades com nomes "tricky" ===\n')

tricky_names = ['Pirassununga', 'Luís Antônio', "Lambari D'Oeste",
                'São José dos Campos', 'Goiânia']
for name in tricky_names:
    matches = cw[cw['municipio'].str.contains(name, na=False, regex=False)]
    if len(matches) > 0:
        r = matches.iloc[0]
        print(f'  {r.municipio:30s} | {r.uf} | muni_key={r.muni_key}')

## Salvar em data/interim/

In [ ]:
# Salva o crosswalk em data/interim/
save_crosswalk(cw)

# Confirma que foi salvo + releitura para verificar integridade
from pipeline.config import interim
out = interim('crosswalk_centrosul.csv')
print(f'\nTamanho do arquivo: {out.stat().st_size / 1024:.1f} KB')
print(f'\nReleitura para verificar integridade:')
cw_reread = pd.read_csv(out, dtype={'geocode': str})
print(f'Shape: {cw_reread.shape}')
print(f'Geocode é string: {cw_reread["geocode"].dtype}')
print(f'Geocode mantém zfill: {cw_reread["geocode"].iloc[0]!r} '
      f'(len={len(cw_reread["geocode"].iloc[0])})')

## Resumo

Se a validação passou e o arquivo foi salvo, próxima camada:

**`02_anp.ipynb`** — adapta os 5 ajustes cirúrgicos do `renovabio_v2.py`:
1. Sem ES nas UFs
2. Doses T2 (vol) e T3 (NEEA) separadas
3. Média ponderada por volume
4. n_usinas_baseline (até 2019, não time-varying)
5. Tratamento staggered via g_m promovido a principal